In [ ]:
import pandas as pd
import pypsa
import yaml

In [ ]:
from _helpers_notebooks import mock_snakemake
snakemake = mock_snakemake(
            "prepare_steel_demand",
            cost_year = 2050,
    )

In [ ]:
cost_year = snakemake.wildcards.cost_year

### File paths

In [ ]:
demand_fn = snakemake.input.steel_demand
bus_locations_fn = snakemake.input.bus_locations

# Outputs
demand_clustered_fn = snakemake.output.steel_demand

# config
config = snakemake.config

### Get steel data and select the year

In [ ]:
# Read the CSV data
demand = pd.read_csv(demand_fn, index_col='iso3')

In [ ]:
# Select the cost_year and scenario
demand = demand[demand['Year'] == int(cost_year)]
demand = demand[demand['scenario'] == 'PkBudg650']
demand = demand.drop(columns=['Year', 'scenario', "remind_region"])

In [ ]:
demand["dri_route"] = demand["raw_steel_demand_mt"] * demand["DRI_Share"]

In [ ]:
# Rename the column to 'value' and and format to Mt instead of kg
demand['dri_route'] = demand['dri_route'].div(1e9)  # Convert from kg to Mt
demand = demand.rename(columns={'dri_route': 'SteelDemand_DRI_Mt'})

In [ ]:
demand.loc["CHN"]

### Cluster per region

### Cluster countries per region
Map each country in the production data to its region using the region definitions from the config file.

In [ ]:
# Load region definitions from config file
regions = config['regions']

In [ ]:
# Build a mapping from ISO-3 code to region (config regions now use ISO-3 codes directly)
iso_to_region = {}
for region, iso_codes in regions.items():
    for code in iso_codes:
        iso_to_region[code] = region

In [ ]:
# Map each row in production to its region
demand['region'] = demand.index.map(lambda iso: iso_to_region.get(iso, 'Other'))
demand.sort_values(by='SteelDemand_DRI_Mt', ascending=False)

### Aggregate steel production by region

In [ ]:
# Group by region and sum all numeric columns
demand_by_region = demand.groupby('region').sum(numeric_only=True)
demand_by_region.sort_values(by='SteelDemand_DRI_Mt', ascending=False)

In [ ]:
# demand_by_region.sum()

### Save clustered steel production

In [ ]:
demand_by_region.to_csv(demand_clustered_fn)